## Doc2Vec

__Что такое Doc2Vec?__  
Doc2Vec — это расширение Word2Vec, которое используется для создания векторных представлений не только слов, но и целых документов. Основная идея — обучить модель, чтобы похожие документы имели близкие векторы.
Модель Doc2Vec, в отличие от модели Word2Vec, используется для создания векторизованного представления группы слов, взятых в совокупности как единое целое. Она не просто вычисляет среднее значение слов в предложении.  


### I. Импорт

In [1]:
import gensim
import gensim.downloader as api

### II. Data - > <font color='yellow'>data (type - list)</font>

In [2]:
dataset = api.load("text8")
data = [d for d in dataset]

[==================================================] 100.0% 31.6/31.6MB downloaded


In [13]:
type(dataset)

text8.Dataset

In [3]:
type(data)

list

In [4]:
len(data)

1701

In [ ]:
data[5]

In [10]:
len(data[4])

10000

### III. Подготовка данных для обучения модели Doc2Vec -><font color='yellow'>data_for_training</font>

Функция __tagged_document__ является генератором, который преобразует список списков слов в объекты TaggedDocument из библиотеки Gensim. Эти объекты используются для обучения модели Doc2Vec.  

<u>Итерация по документам:</u>  
Цикл __for i, list_of_words in enumerate(list_of_list_of_words)__ перебирает каждый список слов (документ) вместе с его индексом i.  

<u>Создание объекта TaggedDocument:</u>  
Для каждого документа вызывается __gensim.models.doc2vec.TaggedDocument__, который принимает два аргумента:  
- __list_of_words:__ список слов из текущего документа.  
- __[i]:__ список тегов, которые идентифицируют документ. В данном случае тегом выступает индекс документа i.

<u>Использование yield:</u>  
Вместо return используется __yield,__ чтобы функция возвращала генератор. Это позволяет обрабатывать большие объёмы данных по одному документу за раз, без необходимости держать все данные в памяти.  

<u>Зачем это нужно?</u>  
Функция подготавливает данные для обучения модели Doc2Vec. Модель <big>_Doc2Vec_</big>, в отличие от <big>_Word2Vec_</big>, работает с "документами" (то есть последовательностями слов) и требует, чтобы каждый документ был помечен уникальным тегом.

<u>TaggedDocument</u> - структура данных в Gensim, которая связывает документ (список слов) с тегом (идентификатором документа).

In [16]:
def tagged_document(list_of_list_of_words):
    for i, list_of_words in enumerate(list_of_list_of_words):
        yield gensim.models.doc2vec.TaggedDocument(list_of_words, [i])

In [17]:
data_for_training = list(tagged_document(data))

In [20]:
print(data_for_training [1:2])

[TaggedDocument(words=['reciprocity', 'qualitative', 'impairments', 'in', 'communication', 'as', 'manifested', 'by', 'at', 'least', 'one', 'of', 'the', 'following', 'delay', 'in', 'or', 'total', 'lack', 'of', 'the', 'development', 'of', 'spoken', 'language', 'not', 'accompanied', 'by', 'an', 'attempt', 'to', 'compensate', 'through', 'alternative', 'modes', 'of', 'communication', 'such', 'as', 'gesture', 'or', 'mime', 'in', 'individuals', 'with', 'adequate', 'speech', 'marked', 'impairment', 'in', 'the', 'ability', 'to', 'initiate', 'or', 'sustain', 'a', 'conversation', 'with', 'others', 'stereotyped', 'and', 'repetitive', 'use', 'of', 'language', 'or', 'idiosyncratic', 'language', 'lack', 'of', 'varied', 'spontaneous', 'make', 'believe', 'play', 'or', 'social', 'imitative', 'play', 'appropriate', 'to', 'developmental', 'level', 'restricted', 'repetitive', 'and', 'stereotyped', 'patterns', 'of', 'behavior', 'interests', 'and', 'activities', 'as', 'manifested', 'by', 'at', 'least', 'one'

### IV. Создание и настройка модели Doc2Vec -> <font color='yellow'>model</font>

__gensim.models.doc2vec.Doc2Vec:__
Конструктор модели Doc2Vec, который создаёт пустую модель с заданными параметрами. После этого модель должна быть обучена на данных.

__Параметры:__
<u>vector_size=40:</u>  
Указывает размерность векторного пространства, в котором будут представлены документы.
Например, если vector_size=40, то каждый документ будет представлен как вектор длиной 40.  
<u>min_count=2:</u>  
Указывает минимальное количество вхождений слова в документах, чтобы оно было включено в словарь модели.
Слова, которые встречаются реже, игнорируются. Это помогает отсеивать шумовые данные.  
<u>epochs=30:</u>  
Количество эпох (итераций) обучения модели. Чем больше эпох, тем дольше обучение, но тем качественнее могут быть векторы, если данных достаточно.

In [21]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=40, min_count=2, epochs=30)
# Или:
# from gensim.models import Doc2Vec
# model = Doc2Vec(vector_size=40, min_count=2, epochs=30)

### V. Обучение модели

In [22]:
# Создаём словарь
model.build_vocab(data_for_training) 

In [23]:
model.train(data_for_training, total_examples=model.corpus_count, epochs=model.epochs)

__train__  
- Обновление весов модели: Во время обучения Doc2Vec обновляет веса модели, чтобы оптимизировать векторные представления слов и документов.  
- Обработка каждого документа: Для каждого документа анализируются:
    - Связи между словами.
    - Связи между словами и тегами документа. Это помогает обучить модель так, чтобы документы с похожим содержанием имели близкие векторы.

__Аргументы функции__  
- data_for_training:  
    это данные, на которых обучается модель.
Обычно это список объектов TaggedDocument, содержащих текст документа и его уникальный тег.
Данные должны быть подготовлены перед обучением, например, с использованием gensim.models.doc2vec.TaggedDocument.  
- total_examples=model.corpus_count:  
    указывает общее количество документов (примеров) в наборе данных для обучения.
- model.corpus_count:  
    это свойство модели, которое содержит количество документов, добавленных в словарь через build_vocab.
- epochs=model.epochs:  
указывает, сколько раз модель должна пройти по всему набору данных (data_for_training).
model.epochs — это параметр, заданный при создании модели (например, epochs=30).

### VI. Использование результатов

#### 1. Сходство нового документа с документами из базы

In [36]:
# У нас новый документ. Получим его вектор
new_vector = model.infer_vector(['violent', 'means', 'to', 'destroy', 'the','organization'])
print(new_vector)

[-0.03390574 -0.09077515 -0.40299907  0.20557    -0.0005819  -0.08686664
 -0.22001784 -0.11802072 -0.25210544  0.03138487  0.17958887  0.00724517
 -0.11203653  0.0495389  -0.2566204   0.00834966  0.08223968  0.07329832
 -0.31890038 -0.09730119  0.05164008  0.03635643 -0.02502169 -0.01646833
 -0.07960222 -0.04561639 -0.25920796 -0.20966704  0.04614365 -0.13518946
  0.11705586  0.2551458  -0.26921383 -0.429616   -0.11410689  0.00340828
 -0.09260072 -0.17043625 -0.13474914 -0.2937331 ]


__Вектор нового документа,__ полученный с помощью модели Doc2Vec, представляет собой числовое описание этого документа в многомерном пространстве, которое отражает его семантическое содержание.  
Вектор нового документа __позволяет:__  
- сравнить документы:  
    если два документа имеют похожие векторы, то их содержание близко по смыслу.  
- вычислить сходство:  
    можно использовать косинусное сходство или евклидову метрику для измерения схожести нового документа с другими документами в базе.

In [37]:
# Оцениваем сходство
# (индексы/теги и степени сходства с документами в обученной модели.)
similarities = model.dv.most_similar([new_vector], topn=5)
print("Похожие документы:", similarities)

Похожие документы: [(93, 0.6505834460258484), (1670, 0.6221058964729309), (1313, 0.6135939359664917), (1697, 0.6134842038154602), (883, 0.610503077507019)]


#### 2. Кластеризация

##### 2.1. Сама кластеризация

- разделение документов по темам;  
- группировка по смыслу: Похожие документы будут находиться в одном кластере.

In [38]:
len(model.dv)

1701

In [43]:
from sklearn.cluster import KMeans
import os

In [ ]:
# ________________
# Для того, чтобы избежать предупреждения - UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
os.environ["OMP_NUM_THREADS"] = "7"  # Укажите количество потоков (7 — примерное значение)
# __________________
# Получаем векторы всех документов
doc_vectors = [model.dv[i] for i in range(len(model.dv))]
# Кластеризация
kmeans = KMeans(n_clusters=3, random_state=0)
clusters = kmeans.fit_predict(doc_vectors) # Получаем метки кластеров для документов
print("Кластеры документов:", clusters)


Кластеры документов: [2 1 0 ... 0 0 1]


c:\Users\shaps\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(


#### 2.2. Сопоставление документов с их кластерной меткой

In [50]:
# Сопоставляем каждый документ с его кластером
documents_with_clusters = [(i, clusters[i]) for i in range(len(clusters ))]

In [51]:
type(documents_with_clusters)

list

In [55]:
documents_with_clusters[0]


(0, 2)

In [54]:
type(documents_with_clusters[0])


tuple

In [57]:
# Какие документы относятся к кластеру 0
cluster_0_docs = [doc_id for doc_id, cluster in documents_with_clusters if cluster == 0]
len(cluster_0_docs)

452

In [ ]:
# Отобразим документы в кластере 0
for doc_id in cluster_0_docs:
    print(f"Документ {doc_id} в кластере 0: {' '.join(data[doc_id])}")

#### 2.3. Визуализация распределения кластеров

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Снижаем размерность до 2D для визуализации
pca = PCA(n_components=2)
reduced_vectors = pca.fit_transform(doc_vectors)

# Рисуем
for cluster in range(3):  # Для каждого кластера
    cluster_points = reduced_vectors[clusters == cluster]
    plt.scatter(cluster_points[:, 0], cluster_points[:, 1], label=f"Кластер {cluster}")

plt.legend()
plt.title("Распределение документов по кластерам")
plt.show()

#### 2.4. Оценка качества кластеризации

##### ___________ Внутренние метрики

        a. Силуэтный коэффициент (Silhouette Score)

Диапазон: от -1 до 1.
- Значения, близкие к 1, означают, что кластеры чётко разделены.
- Значения около 0 означают, что кластеры перекрываются.
- Значения ниже 0 говорят о неправильной кластеризации.

In [63]:
type(clusters)

numpy.ndarray

In [64]:
from sklearn.metrics import silhouette_score

silhouette_avg = silhouette_score(doc_vectors, clusters)
print("Силуэтный коэффициент:", silhouette_avg)

Силуэтный коэффициент: 0.047013745


очень маленький, классы перекрываются

    б. Индекс Дависа-Болдина (Davis-Bouldin Index)

Измеряет среднее отношение расстояний внутри кластеров к расстояниям между кластерами. Диапазон: от 0 и выше. Чем меньше значение, тем лучше.

In [65]:
from sklearn.metrics import davies_bouldin_score

db_score = davies_bouldin_score(doc_vectors, clusters)
print("Индекс Дависа-Болдина:", db_score)


Индекс Дависа-Болдина: 3.7755568507105015


индекс 3.775 - ?

    в. Инерция (Inertia)

Мера компактности кластеров, используемая KMeans. Чем меньше значение, тем лучше (но слишком маленькая инерция может означать переобучение).

In [66]:
print("Инерция кластеров:", kmeans.inertia_)

Инерция кластеров: 231287.61259720282


231287 - вероятно это очень большое значение

##### ___________ Внешние метрики


Используются, если есть истинные метки (true_labels) для документов.
-  Adjusted Rand Index (ARI)
-  Normalized Mutual Information (NMI)
-  Fowlkes-Mallows Index